In [11]:
#Encoding
categorical_columns = X.select_dtypes(include=['object']).columns
numerical_columns = X.select_dtypes(include=['float64', 'int64']).columns

# One-hot encode categorical features using get_dummies
X_encoded = pd.get_dummies(X, columns=categorical_columns, drop_first=True, prefix_sep='_')#drop_first=False

# Concatenate numerical and one-hot encoded categorical columns
X_combined = pd.concat([X[numerical_columns], X_encoded], axis=1)


In [14]:
# Splitting the data into train, test, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)



In [32]:
# Focal Loss Model
def score_eval_func(y_test, y_pred_proba):
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    return roc_auc, pr_auc

# Instantiate and fit the focal loss model
xgbooster_focal = imb_xgb(special_objective='focal', focal_gamma=2.0)
xgbooster_focal.fit(X_train_selected, y_train)

# Predict and evaluate
y_pred_prob_focal = xgbooster_focal.predict_two_class(X_test_selected, y=None)
roc_auc_focal, pr_auc_focal = score_eval_func(y_test, y_pred_prob_focal[:, 1])
print(f"Precision-Recall AUC: {pr_auc_focal:.4f}")
table.add_row(["Focal Loss", f"{roc_auc_focal:.4f}", f"{pr_auc_focal:.4f}"])

Precision-Recall AUC: 0.0023


In [33]:
# Weighted Loss Model
xgbooster_weight = imb_xgb(special_objective='weighted', imbalance_alpha=2.0)
xgbooster_weight.fit(X_train_selected, y_train)

# Predict and evaluate
y_pred_prob_weight = xgbooster_weight.predict_two_class(X_test_selected, y=None)
roc_auc_weight, pr_auc_weight = score_eval_func(y_test, y_pred_prob_weight[:, 1])
print(f'Precision-Recall AUC: {pr_auc_weight:.4f}')
table.add_row(["Weighted Loss", f"{roc_auc_weight:.4f}", f"{pr_auc_weight:.4f}"])

Precision-Recall AUC: 0.0022
